# Analyze SINDy Autoencoder - Pendulum

Loads a trained checkpoint and checks reconstruction / SINDy error, matching the original `analyze_pendulum_model*.ipynb` notebooks.

In [ ]:
import os, sys

# Path to the SindyAutoencoders_Torch folder (this repo). If you uploaded/cloned
# it into Colab's local disk or mounted it from Google Drive, set this to that
# location - e.g. '/content/SindyAutoencoders_Torch' or
# '/content/drive/MyDrive/SindyAutoencoders_Torch'.
REPO_ROOT = '/content/SindyAutoencoders_Torch'

if not os.path.isdir(REPO_ROOT):
    # fall back to running locally, from within this notebook's own example folder
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

EXAMPLE_DIR = os.path.join(REPO_ROOT, 'examples', 'pendulum')
os.chdir(EXAMPLE_DIR)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, EXAMPLE_DIR)
print('Working directory:', os.getcwd())


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import odeint

from data import get_pendulum_data, pendulum_to_movie
from model import SindyAutoencoder
from sindy_utils import sindy_simulate_order2


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('No GPU detected - in Colab, check Runtime > Change runtime type > GPU.')


In [ ]:
save_name = 'model1'  # change to whatever checkpoint you trained

ckpt = torch.load(save_name + '.pt', map_location=device, weights_only=False)
params = ckpt['params']
model = SindyAutoencoder(params).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def evaluate(data):
    batch = {k: torch.as_tensor(data[k], dtype=torch.float32, device=device)
             for k in ('x', 'dx', 'ddx') if k in data}
    with torch.no_grad():
        out = model(batch)
    return {k: v.detach().cpu().numpy() for k, v in out.items()}


## Multiple initial conditions: identified vs. true phase portrait

In [ ]:
t = np.arange(0, 20, .02)
z0s = np.pi/np.array([1.5, 2, 3, 4, 8, 16])
dz0s = .5*np.ones(z0s.shape)
f = lambda z, t: [z[1], -np.sin(z[0])]
n_ics = z0s.size

z = np.zeros((n_ics, t.size, 2))
dz = np.zeros(z.shape)
for i in range(n_ics):
    z[i] = odeint(f, [z0s[i], dz0s[i]], t)
    dz[i] = np.array([f(z[i, j], t[j]) for j in range(len(t))])

x, dx, ddx = pendulum_to_movie(z, dz)

test_data = {}
test_data['x'] = x.reshape((-1, params['input_dim']))
test_data['dx'] = dx.reshape((-1, params['input_dim']))
test_data['ddx'] = ddx.reshape((-1, params['input_dim']))
test_data['z'] = z[:, :, 0].reshape((-1, params['latent_dim']))
test_data['dz'] = z[:, :, 1].reshape((-1, params['latent_dim']))
test_data['ddz'] = dz[:, :, 1].reshape((-1, params['latent_dim']))

results = evaluate(test_data)


In [ ]:
true_coefficients = np.zeros(results['sindy_coefficients'].shape)
true_coefficients[-2] = -1.

z_sim = np.zeros((n_ics, t.size, 2))
pendulum_sim = np.zeros(z_sim.shape)
for i in range(n_ics):
    z_sim[i] = sindy_simulate_order2(
        results['z'][i*t.size], results['dz'][i*t.size], t,
        params['coefficient_mask']*results['sindy_coefficients'],
        params['poly_order'], params['include_sine'])
    pendulum_sim[i] = sindy_simulate_order2(
        test_data['z'][i*t.size], test_data['dz'][i*t.size], t,
        true_coefficients, params['poly_order'], params['include_sine'])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].plot(z_sim[:, :, 0].T, z_sim[:, :, 1].T, linewidth=1)
axes[0].set_title('Identified model'); axes[0].axis('equal')
axes[1].plot(pendulum_sim[:, :, 0].T, pendulum_sim[:, :, 1].T, linewidth=1)
axes[1].set_title('True pendulum'); axes[1].axis('equal')
plt.show()

sim_error = np.mean((z_sim - pendulum_sim)**2)/np.mean(pendulum_sim**2)
print('Identified vs. true pendulum-dynamics relative error: %f' % sim_error)


## Held-out trajectories: reconstruction / SINDy error

In [ ]:
test_data = get_pendulum_data(10)
results = evaluate(test_data)

decoder_x_error = np.mean((test_data['x'] - results['x_decode'])**2)/np.mean(test_data['x']**2)
decoder_ddx_error = np.mean((test_data['ddx'] - results['ddx_decode'])**2)/np.mean(test_data['ddx']**2)
sindy_ddz_error = np.mean((results['ddz'] - results['ddz_predict'])**2)/np.mean(results['ddz']**2)

print('Decoder relative error: %f' % decoder_x_error)
print('Decoder relative SINDy error: %f' % decoder_ddx_error)
print('SINDy relative error, z: %f' % sindy_ddz_error)
